# Introduction

This notebook contains an analysis of water samples for various viruses. The focus is on the taxonomy and Baltimore classification of these viruses. This data is compared with weather data. The weather data consists of the average values from the five days preceding the day the sample was collected at each location.

# Imports

In [10]:
# For data import
from pathlib import Path # for handling file paths
import re # for regular expressions to create valid dataframe names

# For data merging
import os # for file and directory operations

import pandas as pd # for data manipulation
import numpy as np # for numerical operations
import seaborn as sns # for data visualization
import matplotlib.pyplot as plt # for plotting
from skbio.diversity.alpha import shannon # for calculating alpha diversity
from skbio.diversity import beta_diversity # for calculating beta diversity
from skbio.stats.ordination import pcoa # for principal coordinates analysis
from scipy.stats import pearsonr # for correlation analysis

# Data Import

## Load Samplings

In [11]:
def find_folder(*path_parts):
    possible_folders = [
        Path(*path_parts),
        Path("..") / Path(*path_parts),
    ]

    for folder in possible_folders:
        if folder.exists():
            return folder

    raise FileNotFoundError(f"Folder not found: {'/'.join(path_parts)}")


def load_merged_reads_dataframes(folder_path):
    created_dataframes = {}

    for csv_file in sorted(Path(folder_path).glob("*merged_reads.csv")):
        dataframe_name = "df_" + re.sub(r"\W|^(?=\d)", "_", csv_file.stem).lower()
        df = pd.read_csv(csv_file)

        created_dataframes[dataframe_name] = df

    return created_dataframes


samplings_dir = find_folder("data", "samplings")
merged_reads_dataframes = load_merged_reads_dataframes(samplings_dir)

print("Merged reads DataFrames:", merged_reads_dataframes)

Merged reads DataFrames: {'df_copenhagen_merged_reads':                                  name    taxid  ERR14789322  ERR14789323  \
0                      Mastadenovirus    10509          NaN          NaN   
1                          Sequivirus    12057          NaN          NaN   
2                         Sobemovirus    12137          NaN          NaN   
3                         Tombusvirus    12141          NaN          NaN   
4                           Tymovirus    12148          NaN          NaN   
..                                ...      ...          ...          ...   
225                      Baldwinvirus  3153089          NaN          NaN   
226                        Hodnevirus  3153200          NaN          NaN   
227                        Risoevirus  3424972          NaN          NaN   
228                     Margaeryvirus  3425048          NaN          NaN   
229                      Nicoomyvirus  3425078          NaN         0.18   

     ERR14789324  ERR14789325  

## Load Viruses Data

In [ ]:
def load_viruses_cleaned_dataframe(folder_path):
    csv_file = Path(folder_path) / "viruses_cleaned.csv"
    return pd.read_csv(csv_file)


viruses_dir = find_folder("data", "viruses")
df_viruses = load_viruses_cleaned_dataframe(viruses_dir)

print("df_viruses:", df_viruses.shape)

df_viruses: (42124, 12)


# Merge Data

In this step, the virus data is appended to the DataFrames for each city.

In [13]:
# Get columns of df_viruses
df_viruses_columns = df_viruses.columns.tolist()
print("Columns in df_viruses:", df_viruses_columns)

Columns in df_viruses: ['virus tax id', 'host tax id', 'host name', 'realm', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'baltimore_class']


In [14]:
# Merge one DataFrame with df_viruses by the 'name' column.
def merge_with_viruses(df, df_viruses):
    # Add all virus metadata columns to the original DataFrame
    merged_df = df.copy()

    # Add new columns for virus metadata, initialized with NaN
    for col in df_viruses_columns:
        merged_df[col] = np.nan
    
    # Identify viruses from df with matching 'genus' or 'species' in df_viruses
    # For each row in df['taxid'], check if it matches any 'genus' or 'species' in df_viruses and add the corresponding virus metadata to the new columns.
    for virus in df['taxid']:
        virus_info = df_viruses[(df_viruses['virus tax id'] == virus)]
        if not virus_info.empty:
            for col in df_viruses_columns:
                merged_df.loc[merged_df['taxid'] == virus, col] = virus_info.iloc[0][col]

    return merged_df

In [15]:
# Function to clear all files and subdirectories in a given directory
def clear_directory(directory):
    if not os.path.exists(directory):
        return

    for root, dirs, files in os.walk(directory, topdown=False):
        for name in files:
            os.remove(os.path.join(root, name))
        for name in dirs:
            os.rmdir(os.path.join(root, name))

In [16]:
output_folder = Path("../data/samplings_with_viruses_information")
    
## Create directories if they don't exist
os.makedirs(output_folder, exist_ok=True)
    
## If the data is already downloaded, delete it to ensure we have the latest version
clear_directory(output_folder)


merged_reads_dataframes = load_merged_reads_dataframes(samplings_dir)

# Merge all merged reads DataFrames with df_viruses by merge_with_viruses function and store the results in a dictionary and in a folder

merged_dataframes = {}

for name, df in merged_reads_dataframes.items():
    merged_df = merge_with_viruses(df, df_viruses)
    
    merged_dataframes[name] = merged_df

    # Save the merged DataFrame to a new CSV file
    output_file = output_folder / f"{name}_merged.csv"
    merged_df.to_csv(output_file, index=False)
    
    
    

# General Analysis of Measured Viruses

# Completeness of Virus Data

In [17]:
# Check how many rows in each merged DataFrame have and have not virus information (i.e., non-null values in the virus metadata columns).
for dataframe_name in merged_with_viruses_dataframes:
    df = globals()[dataframe_name]
    
    virus_metadata_columns = [column for column in df.columns if column not in ['name', 'genus', 'species']]
    has_virus_info = df[virus_metadata_columns].notnull().any(axis=1)
    print(f"{dataframe_name}: {has_virus_info.sum()} rows with virus info, {len(df) - has_virus_info.sum()} rows without virus info")

NameError: name 'merged_with_viruses_dataframes' is not defined

In [ ]:
cities = {
    'Copenhagen':   ('Copenhagen_merged_reads.csv',  'Copenhagen_weather.csv'),
    'Guangzhou':    ('Guangzhou_merged_reads.csv',   'Guangzhou_weather.csv'),
    'Kuala Lumpur': ('KualaLumpur_merged_reads.csv', 'KualaLumpur_weather.csv'),
    'Melbourne':    ('Melbourne_merged_reads.csv',   'Melbourne_weather.csv'),
    'Quito':        ('Quito_merged_reads.csv',       'Quito_weather.csv'),
    'Regina':       ('Regina_merged_reads.csv',      'Regina_weather.csv'),
    'Seattle':      ('Seattle_merged_reads.csv',     'Seattle_weather.csv'),
    'Yaounde':      ('Yaounde_merged_reads.csv',     'Yaounde_weather.csv'),
}


wetter_spalten = {
    'temperature_2m_mean (°C)':      'Temp',
    'rain_sum (mm)':                 'Regen',
    'relative_humidity_2m_mean (%)': 'Luftfeuchtigkeit',
}

print("Lade ENA-Metadaten...")
ena_data = pd.read_csv('ena_data.tsv', sep='\t')
print(f"ENA-Daten: {len(ena_data)} Samples\n")


# Daten laden, berechnen, speichern 

alle_daten = {}
alle_virus  = {}
alle_pcoa   = {}

for city, (virus_file, weather_file) in cities.items():

    # Virus-Tabelle laden (Zeilen = Samples, Spalten = Virusarten)
    virus_df = pd.read_csv(virus_file, index_col=0).fillna(0).T
    virus_df.columns = virus_df.columns.str.strip()
    virus_df = virus_df.drop('taxid', errors='ignore')

    # Shannon-Diversität pro Sample
    alpha_div = pd.DataFrame([
        {'Sample': sid, 'Shannon': shannon(virus_df.loc[sid].values)}
        for sid in virus_df.index
    ])

    # Datum ENA-Metadaten holen
    ena_city = (
        ena_data[ena_data['run_accession'].isin(alpha_div['Sample'])]
        [['run_accession', 'collection_date']]
        .copy()
        .rename(columns={'run_accession': 'Sample', 'collection_date': 'Date'})
    )
    ena_city['Date'] = pd.to_datetime(ena_city['Date'])

    data = alpha_div.merge(ena_city, on='Sample')

    # Wetterdaten laden
    weather_df = pd.read_csv(weather_file)
    weather_df['time'] = pd.to_datetime(weather_df['time'])

    # Mittelwert plus 4 Tage vor Sammeldatum
    for csv_spalte, col_name in wetter_spalten.items():
        data[col_name] = np.nan
        for i, row in data.iterrows():
            start = row['Date'] - pd.Timedelta(days=4)
            maske = (weather_df['time'] >= start) & (weather_df['time'] <= row['Date'])
            data.at[i, col_name] = weather_df.loc[maske, csv_spalte].mean()

    data_clean = data.dropna(subset=list(wetter_spalten.values()) + ['Shannon'])

    # Beta-Diversität (Bray-Curtis) und PCoA berechnen
    bc_dm    = beta_diversity('braycurtis', virus_df.values, virus_df.index)
    pcoa_res = pcoa(bc_dm)

    alle_daten[city] = data_clean
    alle_virus[city] = virus_df
    alle_pcoa[city]  = pcoa_res

gesamt_df = pd.concat(alle_daten.values(), ignore_index=True)
grenzen = {
    'Temp':             (gesamt_df['Temp'].min() - 2,             gesamt_df['Temp'].max() + 2),
    'Regen':            (gesamt_df['Regen'].min() - 2,            gesamt_df['Regen'].max() + 2),
    'Luftfeuchtigkeit': (gesamt_df['Luftfeuchtigkeit'].min() - 2, gesamt_df['Luftfeuchtigkeit'].max() + 2),
    'Shannon':          (gesamt_df['Shannon'].min() - 0.2,        gesamt_df['Shannon'].max() + 0.2),
}


# Plots


plot_config = [
    ('Temp',             'RdYlBu_r', 'Temperatur (°C)'),
    ('Regen',            'Blues',    'Regen (mm)'),
    ('Luftfeuchtigkeit', 'Blues',    'Luftfeuchtigkeit (%)'),
]

n_cities = len(cities)
fig, axes = plt.subplots(n_cities * 2, 3, figsize=(20, 8 * n_cities))

for idx, city in enumerate(cities.keys()):
    try:
        data_clean = alle_daten[city]
        virus_df   = alle_virus[city]
        pcoa_res   = alle_pcoa[city]

        print(f"{city}: {len(virus_df)} Samples, {len(virus_df.columns)} Viren")

        pc1 = pcoa_res.samples['PC1']
        pc2 = pcoa_res.samples['PC2']

        # Wetterwerte Reihenfolge PCoA-Samples zuordnen
        wetter = (
            virus_df.index.to_frame(name='Sample')
            .merge(data_clean[['Sample'] + list(wetter_spalten.values())], on='Sample', how='left')
        )

        # PCoA-Plot, eingefärbt nach Wetter-Variable
        for col_idx, (col, cmap, label) in enumerate(plot_config):
            vmin, vmax = grenzen[col]
            sc = axes[idx, col_idx].scatter(
                pc1, pc2, s=150, c=wetter[col], cmap=cmap,
                vmin=vmin, vmax=vmax, alpha=0.7, edgecolors='black', linewidth=1
            )
            axes[idx, col_idx].set_xlabel(f'PC1 ({pcoa_res.proportion_explained[0]:.1%})')
            axes[idx, col_idx].set_ylabel(f'PC2 ({pcoa_res.proportion_explained[1]:.1%})')
            axes[idx, col_idx].set_title(f'{city} – PCoA {col}')
            axes[idx, col_idx].grid(True, alpha=0.3)
            plt.colorbar(sc, ax=axes[idx, col_idx], label=label)

        # Korrelation Wetter-Variable vs. Shannon-Diversität
        if len(data_clean) > 2:
            for col_idx, (col, _, label) in enumerate(plot_config):
                r, p = pearsonr(data_clean[col], data_clean['Shannon'])
                sig_text = 'SIGNIFIKANT (p<0.05)' if p < 0.05 else 'nicht signifikant'
                vmin, vmax = grenzen[col]
                ax = axes[idx + n_cities, col_idx]
                ax.scatter(data_clean[col], data_clean['Shannon'],
                           s=150, alpha=0.7, edgecolors='black', linewidth=1, color='steelblue')
                ax.set_xlabel(label)
                ax.set_ylabel('Shannon-Index')
                ax.set_xlim(vmin, vmax)
                ax.set_ylim(*grenzen['Shannon'])
                ax.set_title(f'{city} – {col}\nr={r:.3f} | p={p:.4f} | {sig_text}')
                ax.grid(True, alpha=0.3)

            print(f"  Korrelation Temp–Shannon: r={r:.3f}, p={p:.4f} {sig_text}\n")

    except Exception as e:
        print(f"Fehler bei {city}: {e}\n")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(28, 50))

for idx, city in enumerate(cities.keys()):
    ax = axes[idx // 2, idx % 2]

    virus_city = alle_virus[city]
    daten_city = alle_daten[city]

    # Proben nach Temperatur sortieren
    daten_sorted = daten_city.sort_values('Temp')
    virus_sorted = virus_city.loc[daten_sorted['Sample'].values]

    # Pearson-Korrelation Virus mit Temperatur -> Top 25 
    korrelationen = {
        v: abs(pearsonr(daten_sorted['Temp'], virus_sorted[v])[0])
        for v in virus_sorted.columns
    }
    top25 = sorted(korrelationen, key=korrelationen.get, reverse=True)[:25]

   # Temp zeilen index 
    virus_top25 = virus_sorted[top25].copy()
    virus_top25.index = (daten_sorted['Temp'].round(1).astype(str) + '°C').values

    # relatives Maximum normalisieren (0–1 pro Virus)
    virus_norm = virus_top25.T
    virus_norm = virus_norm.div(virus_norm.max(axis=1), axis=0).fillna(0)

    im = ax.imshow(virus_norm.values, cmap='viridis', aspect='auto')
    plt.colorbar(im, ax=ax, label='Relative Abundanz')
    ax.set_xticks(range(len(virus_norm.columns)))
    ax.set_xticklabels(virus_norm.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(virus_norm.index)))
    ax.set_yticklabels(virus_norm.index)
    ax.set_title(f'Top 25 Viren in {city}')
    ax.set_xlabel('Temperatur (sortiert)')

plt.tight_layout()
plt.show()


In [ ]:
klimagruppen = {
    'Copenhagen':   'Gemäßigt',
    'Melbourne':    'Gemäßigt',
    'Regina':       'Gemäßigt',
    'Seattle':      'Gemäßigt',
    'Guangzhou':    'Subtropisch',
    'Kuala Lumpur': 'Tropisch',
    'Quito':        'Tropisch',
    'Yaounde':      'Tropisch',
}

alle_zusammen = []
for city, data_clean in alle_daten.items():
    df = data_clean[['Temp', 'Shannon']].copy()
    df['Stadt'] = city
    df['Klima'] = klimagruppen[city]
    alle_zusammen.append(df)

gesamt = pd.concat(alle_zusammen, ignore_index=True)

# Alle Klimazonen in einem Plot
sns.lmplot(
    data=gesamt, x='Temp', y='Shannon',
    hue='Klima',
    height=6, aspect=1.5,
    scatter_kws={'alpha': 0.6, 's': 60},
)